In [1]:
"""
Day 2 - Sales Analytics Without Pandas
Squad Python Specialization - Data Science, AI & Analytics

Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.
"""
import csv
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"

### Reading the sales

In [ ]:
def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    with open(path, mode ="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    return rows

In [9]:
rows = read_sales()
print("Number of rows:", len(rows))
print("Column names:", list(rows[0].keys()))
print("First record:", rows[0])

Number of rows: 50
Column names: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
First record: {'sale_id': 'S-3001', 'sale_date': '2026-09-01', 'store': 'Westgate', 'product': 'Safaricom Neon Ray', 'quantity': '1', 'unit_price_kes': '5999', 'payment_method': 'Cash'}


#### Converting and cleaning numeric data safely

In [ ]:
def parse_number(value: str) -> float | None:
    """Convert a messy numeric string to a float, or return None if it cannot be trusted."""

    value = value.strip()

    if not value:
        return None

    if "," in value:
        parts = value.split(".")

        if len(parts) > 2:
            return None

        integer_part = parts[0]
        unsigned_part = integer_part.lstrip("+-")
        groups = unsigned_part.split(",")

        if not groups[0].isdigit() or not (1 <= len(groups[0]) <= 3):
            return None

        if not all(
            len(group) == 3 and group.isdigit()
            for group in groups[1:]):
            return None

        value = value.replace(",", "")

    try:
        number = float(value)

        if number != number or number in (float("inf"), float("-inf")):
            return None

        return number

    except ValueError:
        return None

In [11]:
# Test function
test_values = ["5999", "1,500", "", "N/A", "KES 5999", "12,50"]

for value in test_values:
    print(f"{value!r} -> {parse_number(value)}")

'5999' -> 5999.0
'1,500' -> 1500.0
'' -> None
'N/A' -> None
'KES 5999' -> None
'12,50' -> None


In [ ]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return (clean_rows, skipped_rows). Clean rows have numeric quantity, unit price and revenue."""

    clean_rows = []
    skipped_rows = []

    for row in rows:
        quantity = parse_number(row["quantity"])
        unit_price = parse_number(row["unit_price_kes"])

        # Skip rows with missing or invalid numeric values
        if quantity is None:
            skipped_rows.append({
                **row,
                "reason": "Missing or invalid quantity"})
            continue

        if unit_price is None:
            skipped_rows.append({
                **row,
                "reason": "Missing or invalid unit price"})
            continue

        # Reject zero or negative quantities and prices
        if quantity <= 0:
            skipped_rows.append({
                **row,
                "reason": "Quantity must be greater than zero"})
            continue

        if unit_price <= 0:
            skipped_rows.append({
                **row,
                "reason": "Unit price must be greater than zero"})
            continue

        # Preserve the original fields and add converted numeric values
        clean_row = {
            **row,
            "quantity": quantity,
            "unit_price_kes": unit_price,
            "revenue": quantity * unit_price
        }

        clean_rows.append(clean_row)

    return clean_rows, skipped_rows

In [ ]:
# Test function
rows = read_sales()

clean_rows, skipped_rows = convert_rows(rows)

print("Original rows:", len(rows))
print("Clean rows:", len(clean_rows))
print("Skipped rows:", len(skipped_rows))

print("\nFirst clean record:")
print(clean_rows[0])

print("\nSkipped records:")
for row in skipped_rows:
    print(row["sale_id"], "-", row["reason"])

Original rows: 50
Clean rows: 45
Skipped rows: 5

First clean record:
{'sale_id': 'S-3001', 'sale_date': '2026-09-01', 'store': 'Westgate', 'product': 'Safaricom Neon Ray', 'quantity': 1.0, 'unit_price_kes': 5999.0, 'payment_method': 'Cash', 'revenue': 5999.0}

Skipped records:
S-3006 - Missing or invalid quantity
S-3029 - Missing or invalid unit price
S-3030 - Missing or invalid unit price
S-3035 - Missing or invalid quantity
S-3039 - Missing or invalid unit price


#### Calculating total revenue

In [15]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate the total revenue across all the clean sales"""
    return sum(row["revenue"] for row in rows)

In [16]:
# Testing my total_revenue function
revenue = total_revenue(clean_rows)

print(f"Total Revenue: KES {revenue:,.2f}")

Total Revenue: KES 1,291,813.00


#### Calculating total revenue using dictionaries

In [17]:
def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Calculate total revenue grouped by a specified field."""

    totals = {}

    for row in rows:
        key = row[field]
        revenue = row["revenue"]

        if key not in totals:
            totals[key] = 0.0

        totals[key] += revenue

    return totals

#### Running the three analyses

In [33]:
product_revenue = revenue_by(clean_rows, "product")
store_revenue = revenue_by(clean_rows, "store")
payment_revenue = revenue_by(clean_rows, "payment_method")

print("--- REVENUE BY PRODUCT ---")
for product, revenue in sorted(
    product_revenue.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{product}: KES {revenue:,.2f}")

print("\n--- REVENUE BY STORE ---")
for store, revenue in sorted(
    store_revenue.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{store}: KES {revenue:,.2f}")

print("\n--- REVENUE BY PAYMENT METHOD ---")
for method, revenue in sorted(
    payment_revenue.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{method}: KES {revenue:,.2f}")

--- REVENUE BY PRODUCT ---
HP 15 Laptop: KES 389,994.00
Hisense 43in Smart TV: KES 272,993.00
Lenovo Tab M10: KES 199,992.00
Tecno Spark 20: KES 161,988.00
Infinix Hot 40: KES 63,996.00
JBL Go 3 Speaker: KES 49,491.00
Anker Power Bank 20000: KES 38,691.00
Samsung Galaxy A15: KES 37,998.00
Xiaomi Redmi Buds: KES 25,592.00
Oraimo FreePods: KES 19,593.00
USB-C Charger 25W: KES 19,487.00
Safaricom Neon Ray: KES 11,998.00

--- REVENUE BY STORE ---
Kisumu Mega Plaza: KES 449,678.00
Nairobi CBD: KES 331,085.00
Westgate: KES 235,979.00
Nakuru Westside: KES 139,988.00
Mombasa City Mall: KES 135,083.00

--- REVENUE BY PAYMENT METHOD ---
M-Pesa: KES 738,568.00
Cash: KES 285,376.00
Card: KES 267,869.00


#### Ranking the Top-3 Products

In [19]:
def top_n(totals: dict[str, float], n: int) -> list[tuple[str, float]]:
    """Return the top n items as (name, revenue) tuples."""
    return sorted(
        totals.items(),
        key=lambda item: (-item[1], item[0])
    )[:n]

In [34]:
# Test 
top_products = top_n(product_revenue, 3)

print("-- TOP 3 PRODUCTS BY REVENUE ---")

for rank, (product, revenue) in enumerate(top_products, start=1):
    print(f"{rank}. {product}: KES {revenue:,.2f}")

-- TOP 3 PRODUCTS BY REVENUE ---
1. HP 15 Laptop: KES 389,994.00
2. Hisense 43in Smart TV: KES 272,993.00
3. Lenovo Tab M10: KES 199,992.00


#### Filtering high value Transactions

In [21]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Return sales with revenue strictly greater than the threshold."""
    return [row for row in rows if row["revenue"] > threshold]

In [22]:
# Finding high value sales
threshold = 50_000

high_value_sales = high_value(clean_rows, threshold)

print(f"HIGH-VALUE SALES (Above KES {threshold:,.2f})")
print(f"Number of qualifying sales: {len(high_value_sales)}\n")

for row in high_value_sales:
    print(
        f"{row['sale_id']} | {row['store']} | {row['product']} | "
        f"KES {row['revenue']:,.2f}"
    )

HIGH-VALUE SALES (Above KES 50,000.00)
Number of qualifying sales: 5

S-3013 | Westgate | Lenovo Tab M10 | KES 99,996.00
S-3017 | Kisumu Mega Plaza | Hisense 43in Smart TV | KES 155,996.00
S-3022 | Nairobi CBD | HP 15 Laptop | KES 259,996.00
S-3036 | Nakuru Westside | Tecno Spark 20 | KES 53,996.00
S-3048 | Kisumu Mega Plaza | HP 15 Laptop | KES 129,998.00


In [35]:
# Counting high value sales by store
stores = {row["store"] for row in clean_rows}

high_value_by_store = {
    store: sum(
        1 for row in high_value_sales
        if row["store"] == store
    )
    for store in sorted(stores)
}

print("\n--- HIGH-VALUE SALES BY STORE ---")

for store, count in high_value_by_store.items():
    print(f"{store}: {count}")


--- HIGH-VALUE SALES BY STORE ---
Kisumu Mega Plaza: 2
Mombasa City Mall: 0
Nairobi CBD: 1
Nakuru Westside: 1
Westgate: 1


#### Reusable statistical functions

In [24]:
# Reusable statistics - implement these yourself, no statistics module.
def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("Cannot calculate the mean of an empty list.")

    return sum(values) / len(values)

In [25]:
def median(values: list[float]) -> float:
    if not values:
        raise ValueError("Cannot calculate the median of an empty list.")

    sorted_values = sorted(values)
    count = len(sorted_values)
    middle = count // 2

    if count % 2 == 1:
        return sorted_values[middle]

    return (sorted_values[middle - 1] + sorted_values[middle]) / 2


In [26]:
def minimum_maximum(values: list[float]) -> tuple[float, float]:
    if not values:
        raise ValueError(
            "Cannot calculate minimum and maximum of an empty list."
        )

    smallest = values[0]
    largest = values[0]

    for value in values[1:]:
        if value < smallest:
            smallest = value

        if value > largest:
            largest = value

    return smallest, largest

#### Applying the reusable statistical functions to Transactions Revenue

In [27]:
# Applying the functions to transaction revenue

transaction_revenues = [row["revenue"] for row in clean_rows]

average_revenue = mean(transaction_revenues)
median_revenue = median(transaction_revenues)
minimum_revenue, maximum_revenue = minimum_maximum(transaction_revenues)

print(f"Mean transaction revenue: KES {average_revenue:,.2f}")
print(f"Median transaction revenue: KES {median_revenue:,.2f}")
print(f"Minimum transaction revenue: KES {minimum_revenue:,.2f}")
print(f"Maximum transaction revenue: KES {maximum_revenue:,.2f}")

Mean transaction revenue: KES 28,706.96
Median transaction revenue: KES 10,998.00
Minimum transaction revenue: KES 1,499.00
Maximum transaction revenue: KES 259,996.00


#### Verifying with Pandas

In [ ]:
def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute headline sales results with pandas and compare them."""

    import pandas as pd

    # Read all CSV fields as strings
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    # Apply the  numeric cleaning rules
    df["quantity"] = df["quantity"].map(parse_number)
    df["unit_price_kes"] = df["unit_price_kes"].map(parse_number)

    # Keep only rows with valid, positive quantities and prices
    valid = (
        df["quantity"].notna()
        & df["unit_price_kes"].notna()
        & (df["quantity"] > 0)
        & (df["unit_price_kes"] > 0)
    )

    clean_df = df.loc[valid].copy()

    # Calculate transaction revenue
    clean_df["revenue"] = (
        clean_df["quantity"] * clean_df["unit_price_kes"]
    )

    # Recalculate headline figures
    pandas_results = {
        "clean_rows": len(clean_df),
        "skipped_rows": len(df) - len(clean_df),
        "total_revenue": clean_df["revenue"].sum(),
        "product_revenue": clean_df.groupby("product")["revenue"].sum().to_dict(),
        "store_revenue": clean_df.groupby("store")["revenue"].sum().to_dict(),
        "payment_revenue": clean_df.groupby("payment_method")["revenue"].sum().to_dict(),
        "top_products": sorted(
            clean_df.groupby("product")["revenue"].sum().items(),
            key=lambda item: (-item[1], item[0])
        )[:3],
        "high_value_count": int((clean_df["revenue"] > 50_000).sum()),
        "mean": clean_df["revenue"].mean(),
        "median": clean_df["revenue"].median(),
        "minimum": clean_df["revenue"].min(),
        "maximum": clean_df["revenue"].max(),
    }

    # Compare the standard-library results with pandas
    for key, pandas_value in pandas_results.items():
        python_value = my_results[key]

        if isinstance(pandas_value, dict):
            match = (
                python_value.keys() == pandas_value.keys()
                and all(
                    round(python_value[name], 2) == round(pandas_value[name], 2)
                    for name in pandas_value))
            
        elif key == "top_products":
            match = (
                len(python_value) == len(pandas_value)
                and all(
                    name_a == name_b and round(value_a, 2) == round(value_b, 2)
                    for (name_a, value_a), (name_b, value_b)
                    in zip(python_value, pandas_value)))
            
        elif isinstance(pandas_value, float):
            match = round(python_value, 2) == round(pandas_value, 2)
        else:
            match = python_value == pandas_value

        print(f"{key}: {'MATCH' if match else 'MISMATCH'}")

        if not match:
            print(f"  Python: {python_value}")
            print(f"  pandas: {pandas_value}")

#### Preparing results for comparison

In [29]:
my_results = {
    "clean_rows": len(clean_rows),
    "skipped_rows": len(skipped_rows),
    "total_revenue": total_revenue(clean_rows),
    "product_revenue": product_revenue,
    "store_revenue": store_revenue,
    "payment_revenue": payment_revenue,
    "top_products": top_n(product_revenue, 3),
    "high_value_count": len(high_value_sales),
    "mean": mean(transaction_revenues),
    "median": median(transaction_revenues),
    "minimum": minimum_maximum(transaction_revenues)[0],
    "maximum": minimum_maximum(transaction_revenues)[1],
}

In [30]:
verify_with_pandas(DATA_PATH, my_results)

clean_rows: MATCH
skipped_rows: MATCH
total_revenue: MATCH
product_revenue: MATCH
store_revenue: MATCH
payment_revenue: MATCH
top_products: MATCH
high_value_count: MATCH
mean: MATCH
median: MATCH
minimum: MATCH
maximum: MATCH


In [40]:


# Prepare the report output path
output_path = Path("output") / "sales_report.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Building the report
report_lines = [
    "=" * 60,
    "TECH HUB KENYA - SEPTEMBER 2026 SALES REPORT",
    "=" * 60,
    "",
    "DATA QUALITY",
    "-" * 60,
    f"Original transactions: {len(rows)}",
    f"Clean transactions: {len(clean_rows)}",
    f"Skipped transactions: {len(skipped_rows)}",
    "",
    "TOTAL REVENUE",
    "-" * 60,
    f"KES {total_revenue(clean_rows):,.2f}",
    "",
    "REVENUE BY PRODUCT",
    "-" * 60,
]

# Product revenue, highest first
for product, revenue in sorted(
    product_revenue.items(),
    key=lambda item: (-item[1], item[0])):
    report_lines.append(f"{product}: KES {revenue:,.2f}")

report_lines.extend([
    "",
    "REVENUE BY STORE",
    "-" * 60,])

for store, revenue in sorted(
    store_revenue.items(),
    key=lambda item: (-item[1], item[0])):

    report_lines.append(f"{store}: KES {revenue:,.2f}")

report_lines.extend([
    "",
    "REVENUE BY PAYMENT METHOD",
    "-" * 60,])

for method, revenue in sorted(
    payment_revenue.items(),
    key=lambda item: (-item[1], item[0])):

    report_lines.append(f"{method}: KES {revenue:,.2f}")

report_lines.extend([
    "",
    "TOP 3 PRODUCTS",
    "-" * 60,])

for rank, (product, revenue) in enumerate(top_n(product_revenue, 3), start=1):
    report_lines.append(f"{rank}. {product}: KES {revenue:,.2f}")

report_lines.extend([
    "",
    "HIGH-VALUE TRANSACTIONS",
    "-" * 60,
    "Threshold: strictly greater than KES 50,000.00",
    f"Qualifying transactions: {len(high_value_sales)}",
])

for sale in high_value_sales:
    report_lines.append(
        f"{sale['sale_id']} | {sale['store']} | "
        f"{sale['product']} | KES {sale['revenue']:,.2f}"
    )

report_lines.extend([
    "",
    "HIGH-VALUE TRANSACTIONS BY STORE",
    "-" * 60,
])

for store, count in high_value_by_store.items():
    report_lines.append(f"{store}: {count}")

report_lines.extend([
    "",
    "TRANSACTION REVENUE STATISTICS",
    "-" * 60,
    f"Mean: KES {mean(transaction_revenues):,.2f}",
    f"Median: KES {median(transaction_revenues):,.2f}",
])

minimum_revenue, maximum_revenue = minimum_maximum(transaction_revenues)

report_lines.extend([
    f"Minimum: KES {minimum_revenue:,.2f}",
    f"Maximum: KES {maximum_revenue:,.2f}",
    "",
    "SKIPPED TRANSACTIONS",
    "-" * 60,
])

for sale in skipped_rows:
    report_lines.append(
        f"{sale['sale_id']}: {sale['reason']}"
    )

report_lines.extend([
    "",
    "PANDAS VERIFICATION",
    "-" * 60,
    "See notebook verification output for MATCH/MISMATCH results.",
    "",
])

# writing report to disk
output_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Report saved to: {output_path}")
print(f"Report lines: {len(report_lines)}")

Report saved to: output\sales_report.txt
Report lines: 86
